# Creating Custom Function Nodes

In this tutorial we do a deep dive into function nodes and how users can create their own nodes for various functions.  Users should already be familiar with the concepts covered in the [Sampling Parameters notebook](https://lightcurvelynx.readthedocs.io/en/latest/notebooks/sampling.html).

## Function Nodes

Sampling functions, such as those provided by numpy, are only one type of function that we might want to use to generate parameters. We might want to sample from other functions or apply a mathematical transform to multiple input parameters to compute a new parameter. For example consider the case of computing the `distmod` parameter from `redshift`. We can do this using the information about the cosmology, such as provided by astropy's `FlatLambdaCDM` class:

In [ ]:
from astropy.cosmology import FlatLambdaCDM

cosmo = FlatLambdaCDM(H0=73.0, Om0=0.3)
redshifts = np.array([0.1, 0.2, 0.3])
distmods = cosmo.distmod(redshifts).value
print(distmods)

### Creating New Function Nodes

Importantly, you can use a `FunctionNode` that takes input parameter(s) and produces output parameter(s) during the generation. `FunctionNode` is a subclass of `ParameterizedNode` that wraps the functionality of collecting the inputs, running the computations, and storing the output.

As a simple example, let's create a `FunctionNode` that computes y = m * x + b.

In [ ]:
from lightcurvelynx.base_models import FunctionNode


def _linear_eq(x, m, b):
    """Compute y = m * x + b"""
    return m * x + b


func_node = FunctionNode(
    _linear_eq,  # First parameter is the function to call.
    x=NumpyRandomFunc("uniform", low=0.0, high=10.0),
    m=5.0,
    b=-2.0,
)
print(func_node.sample_parameters(num_samples=1))

The first parameter of the function node is the function to evaluate, such as our linear equation above. Each input into that function **must** be included as a named parameter during the `FunctionNode` definition, such as `x`, `m`, and `b` above. If any of the input parameters are missing the code will give an error.

Here we use constants for `m` and `b` so we use the same linear formulation for each sample. Only the value of `x` changes. However we could have also used function nodes, including sampling functions, to set `m` and `b`. In that case it is important to remember that each of our results sample will be the result of a sampling of all the variable parameters.